# Keras Colab Training Launcher

This notebook is the Colab entry point for training. The real training logic lives in `pipeline/02_train.py`, so the notebook stays small and the project remains reusable from scripts, GitHub, and Colab.

Use this notebook for GPU training. Do not run full training locally.

## 1. Enable GPU

In Colab, go to `Runtime > Change runtime type > T4 GPU`, then run the next cell.

In [1]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Clone The Repo

This checks out the active migration branch.

In [2]:
!git clone https://github.com/SARWAGYASHAH/Anomalous-Sound-Detection-using-Spectrograms.git
%cd Anomalous-Sound-Detection-using-Spectrograms
!git checkout migrate-to-keras
!git pull

Cloning into 'Anomalous-Sound-Detection-using-Spectrograms'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 167 (delta 56), reused 145 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 57.23 KiB | 2.20 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/Anomalous-Sound-Detection-using-Spectrograms
Branch 'migrate-to-keras' set up to track remote branch 'migrate-to-keras' from 'origin'.
Switched to a new branch 'migrate-to-keras'
Already up to date.


## 3. Install Dependencies

In [3]:
# Keep Colab's preinstalled TensorFlow/GPU stack intact.
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4. Attach Data

The raw gearbox zip is stored on Google Drive and downloaded with `gdown`.

If the download cell fails, open the Drive file sharing settings and set access to `Anyone with the link`.

In [4]:
from pathlib import Path

DRIVE_ZIP_URL = 'https://drive.google.com/file/d/1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS/view?usp=drive_link'
ZIP_PATH = Path('Data/dev_data_gearbox.zip')
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    !gdown --fuzzy "$DRIVE_ZIP_URL" -O Data/dev_data_gearbox.zip

print('Zip exists:', ZIP_PATH.exists())
print('Zip size MB:', round(ZIP_PATH.stat().st_size / (1024 * 1024), 2) if ZIP_PATH.exists() else 'missing')

Downloading...
From (original): https://drive.google.com/uc?id=1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS
From (redirected): https://drive.google.com/uc?id=1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS&confirm=t&uuid=231a6f65-5f5c-4add-b3a9-511c3ca4ca39
To: /content/Anomalous-Sound-Detection-using-Spectrograms/Data/dev_data_gearbox.zip
100% 1.20G/1.20G [00:15<00:00, 75.2MB/s]
Zip exists: True
Zip size MB: 1144.64


In [5]:
!unzip -q -o Data/dev_data_gearbox.zip -d Data/
!ls Data
!python pipeline/01_preprocess.py

dev_data_gearbox.zip  gearbox
2026-05-06 17:51:20 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-06 17:51:20 | INFO     | preprocess | Log file: artifacts/logs/run_20260506_175120.log
2026-05-06 17:51:20 | INFO     | preprocess | ============================================================
2026-05-06 17:51:20 | INFO     | preprocess | PIPELINE STAGE 1: Preprocessing (Audio → Spectrograms)
2026-05-06 17:51:20 | INFO     | preprocess | ============================================================
2026-05-06 17:51:20 | INFO     | preprocess | 
Processing split: train
2026-05-06 17:51:20 | INFO     | preprocess |   Input:  Data/gearbox/train
2026-05-06 17:51:20 | INFO     | preprocess |   Output: Data/processed/train
2026-05-06 17:51:20 | INFO     | src.data.audio_loader | Discovered 3026 .wav files in Data/gearbox/train
2026-05-06 17:51:36 | INFO     | preprocess |   Processed 200/3026 files
2026-05-06 17:51:38 | INFO     | preprocess |   Processed 

## 5. Verify Processed Data

In [6]:
from pathlib import Path

for path in [
    Path('Data/processed/train/normal'),
    Path('Data/processed/source_test/normal'),
    Path('Data/processed/source_test/anomaly'),
    Path('Data/processed/target_test/normal'),
    Path('Data/processed/target_test/anomaly'),
]:
    count = len(list(path.glob('*.npy'))) if path.exists() else 0
    print(path, count)

Data/processed/train/normal 3026
Data/processed/source_test/normal 411
Data/processed/source_test/anomaly 351
Data/processed/target_test/normal 309
Data/processed/target_test/anomaly 336


## 6. Dry Run

This builds the dataset/model and runs one forward pass. It does not train.

In [7]:
!python pipeline/02_train.py --dry-run --no-mlflow --batch-size 4

2026-05-06 17:53:32 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-06 17:53:32 | INFO     | train | Log file: artifacts/logs/run_20260506_175332.log
2026-05-06 17:53:32 | INFO     | train | ============================================================
2026-05-06 17:53:32 | INFO     | train | PIPELINE STAGE 2: Training (Keras Autoencoder)
2026-05-06 17:53:32 | INFO     | train | ============================================================
2026-05-06 17:53:32 | INFO     | train | TensorFlow version: 2.20.0
2026-05-06 17:53:32 | INFO     | train | GPU devices: ['/physical_device:GPU:0']
2026-05-06 17:53:32 | INFO     | src.data.dataset | Discovered 3026 spectrograms in Data/processed/train (normal=3026, anomaly=0)
2026-05-06 17:53:32 | INFO     | train | Training data directory: Data/processed/train
2026-05-06 17:53:32 | INFO     | train | Input shape: (128, 313, 1)
2026-05-06 17:53:32 | INFO     | train | Batch size: 4
2026-05-06 17:53:32 | INFO   

## 7. First Short Training Run

Start with a short run to verify Colab paths, GPU, model saving, metadata, and callbacks.

In [8]:
!python pipeline/02_train.py --config config/default.yaml --epochs 3 --no-mlflow

2026-05-06 17:53:50 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-06 17:53:50 | INFO     | train | Log file: artifacts/logs/run_20260506_175350.log
2026-05-06 17:53:50 | INFO     | train | ============================================================
2026-05-06 17:53:50 | INFO     | train | PIPELINE STAGE 2: Training (Keras Autoencoder)
2026-05-06 17:53:50 | INFO     | train | ============================================================
2026-05-06 17:53:50 | INFO     | train | TensorFlow version: 2.20.0
2026-05-06 17:53:50 | INFO     | train | GPU devices: ['/physical_device:GPU:0']
2026-05-06 17:53:50 | INFO     | src.data.dataset | Discovered 3026 spectrograms in Data/processed/train (normal=3026, anomaly=0)
2026-05-06 17:53:50 | INFO     | train | Training data directory: Data/processed/train
2026-05-06 17:53:50 | INFO     | train | Input shape: (128, 313, 1)
2026-05-06 17:53:50 | INFO     | train | Batch size: 32
2026-05-06 17:53:50 | INFO  

## 8. Full Training Run

Run this after the short run succeeds. MLflow is enabled by config unless you pass `--no-mlflow`.

In [9]:
!python pipeline/02_train.py --config config/default.yaml

2026-05-06 17:56:22 | INFO     | src.utils.seed | Random seed set to 42 (random, numpy, tensorflow)
2026-05-06 17:56:22 | INFO     | train | Log file: artifacts/logs/run_20260506_175622.log
2026-05-06 17:56:22 | INFO     | train | ============================================================
2026-05-06 17:56:22 | INFO     | train | PIPELINE STAGE 2: Training (Keras Autoencoder)
2026-05-06 17:56:22 | INFO     | train | ============================================================
2026-05-06 17:56:22 | INFO     | train | TensorFlow version: 2.20.0
2026-05-06 17:56:22 | INFO     | train | GPU devices: ['/physical_device:GPU:0']
2026-05-06 17:56:22 | INFO     | src.data.dataset | Discovered 3026 spectrograms in Data/processed/train (normal=3026, anomaly=0)
2026-05-06 17:56:22 | INFO     | train | Training data directory: Data/processed/train
2026-05-06 17:56:22 | INFO     | train | Input shape: (128, 313, 1)
2026-05-06 17:56:22 | INFO     | train | Batch size: 32
2026-05-06 17:56:22 | INFO  

## 9. Inspect Artifacts

In [10]:
!find artifacts -maxdepth 3 -type f | sort | head -80

artifacts/logs/run_20260506_175120.log
artifacts/logs/run_20260506_175332.log
artifacts/logs/run_20260506_175350.log
artifacts/logs/run_20260506_175622.log
artifacts/metadata/run_20260506_175352.json
artifacts/metadata/run_20260506_175624.json
artifacts/models/v1/best_model.keras
artifacts/models/v1/config_snapshot.yaml
artifacts/models/v1/final_model.keras
artifacts/models/v1/training_log.csv
artifacts/models/v2/best_model.keras
artifacts/models/v2/config_snapshot.yaml
artifacts/models/v2/final_model.keras
artifacts/models/v2/training_log.csv


In [11]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/anomalous_sound_detection/artifacts"
!cp -r artifacts/* "/content/drive/MyDrive/anomalous_sound_detection/artifacts/"


Mounted at /content/drive
